In [ ]:
conda install nbconvert

In [14]:
#import libraries
import numpy as np
import re
import nltk
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from tabulate import tabulate
from collections import Counter
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

#download NLTK data for stop words and inflexions
nltk.download('stopwords' , quiet=True)
nltk.download('wordnet' , quiet=True)

#read CSV file
df = pd.read_csv('customer_complaints_1.csv')

dataset = df['text'].dropna().tolist()

#text pre-processing
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
clean_dataset = []

for doc in dataset:
    #remove punctuations and lowercase
    doc = re.sub(r'[^a-zA-Z\s]','',doc.lower())

    #tokenise, remove stop words and handle inflexions (lemmatise)
    tokens = doc.split()
    clean_tokens = [lemmatizer.lemmatize(word) for word in tokens 
                    if word not in stop_words]
    #rejoin into clean string
    clean_dataset.append("".join(clean_tokens))


#vectorise the dataset
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(clean_dataset)

#perform clustering
k = 3
km = KMeans(n_clusters=k, random_state=42)
km.fit(X) 
y_pred = km.predict(X)

#tabulate the document and predicted cluster
table_data = [["Document (truncated)", "Predicted Cluster"]]
table_data.extend([[doc[:100] + "...", cluster] for doc,
                   cluster in zip(dataset[:20], y_pred[:20])])

print(f"--- TF-IDF Clustering Results (First 20 out of {len(dataset)}complaints) ---")
print(tabulate(table_data, headers="firstrow", tablefmt="grid"))

total_samples = len(y_pred)
cluster_label_counts = [Counter(y_pred)]
purity = sum(max (cluster.values()) for cluster in cluster_label_counts)/ total_samples
print("\nPurity Score:", purity)

    

--- TF-IDF Clustering Results (First 20 out of 19complaints) ---
+---------------------------------------------------------------------------------------------------------+---------------------+
| Document (truncated)                                                                                    |   Predicted Cluster |
+=========================================================================================================+=====================+
| I used to love Comcast. Until all these constant updates. My internet and cable crash a lot at night... |                   0 |
+---------------------------------------------------------------------------------------------------------+---------------------+
| I'm so over Comcast! The worst internet provider. I'm taking online classes and multiple times was l... |                   0 |
+---------------------------------------------------------------------------------------------------------+---------------------+
| If I could give them a 